# 第02章 逼近的智慧：实验

![三格插图：曲线y=x²下的面积先用4块、再用8块梯形近似。](ABC共用-图片/A-章节导入.svg)

*图02-1 · 给弯曲的河岸铺梯形。概念示意，不是实测结果。*

## 先看一幕：给弯曲的河岸铺梯形

弯曲河岸旁的一块地，不方便直接丈量。机器人决定先用小梯形铺一遍，再把每块梯形切细一点：多花一倍功夫，能换来多少精度？

**先猜，再验证：** 对 $\int_0^1 x^2\,dx$，复合梯形公式把区间数从 4 增加到 8，绝对误差会变成原来的几分之一？先写下你的猜想，再与计算对照。

**A组（g02）本次预测（运行前写下）：** 误差变为原来的 $1/4$；对 $f(x)=x^2$ 这一条应当精确成立，因为 $f''$ 为常数。对一般光滑函数只期望渐近成立。

> 状态：A组（g02）编写稿。本 Notebook 的全部输出由 A 组在下方"环境与版本"记录的环境中实际运行得到；B/C 组尚未独立验证。

## 本实验的安排

| 小节 | 问题 | 判据 |
| --- | --- | --- |
| 1 | 环境与版本 | 记录可复现信息 |
| 2 | 梯形公式误差与收敛阶 | $E_n=1/(6n^2)$ 精确成立；观测阶 $p\to2$ |
| 3 | 光滑性失效时的阶 | $|x-1/2|$ 阶仍≈2；$\sqrt{x}$ 掉到≈1.5 |
| 4 | 插值 ≠ 区间上逼近 | 节点误差 $\approx0$ 而区间最大误差随 $n$ 增大 |
| 5 | 换节点：Chebyshev | 最大误差随 $n$ 下降 |
| 6 | 样条与最小二乘 | 与高次插值对比 |
| 7 | 三种求导方式 | 自动微分与符号导数一致到机器精度 |

## 1. 环境与版本

复现需要的信息：Python 与库版本、随机种子、浮点精度。本章实验全部为确定性计算，不使用随机数，因此没有种子；仍记录版本以便 B 组在干净环境比对。

In [1]:
import platform, sys
import numpy as np
import scipy, sympy, matplotlib

import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
from matplotlib.font_manager import findfont, FontProperties

print("python     :", sys.version.split()[0], "|", platform.platform())
print("numpy      :", np.__version__)
print("scipy      :", scipy.__version__)
print("sympy      :", sympy.__version__)
print("matplotlib :", matplotlib.__version__)
print("float64 eps:", np.finfo(float).eps)
print("图注中文字体:", findfont(FontProperties(family=matplotlib.rcParams["font.sans-serif"])))

python     : 3.12.10 | Windows-11-10.0.26200-SP0
numpy      : 1.26.4
scipy      : 1.14.1
sympy      : 1.13.3
matplotlib : 3.9.2
float64 eps: 2.220446049250313e-16
图注中文字体: C:\Windows\Fonts\msyh.ttc


## 2. 梯形公式的误差与收敛阶

复合梯形公式：把 $[a,b]$ 等分成 $n$ 段，$h=(b-a)/n$，

$$T_n(f)=h\Big[\tfrac12 f(x_0)+\sum_{i=1}^{n-1}f(x_i)+\tfrac12 f(x_n)\Big].$$

**参考值（先于运行写下）：** 对 $f(x)=x^2$，$\int_0^1x^2dx=1/3$，正文中推导出 $T_n-1/3=\dfrac{1}{6n^2}$，是**恒等式**而非渐近式。

**容差依据（先紧后松的实际过程）：** 起初用 `rtol=1e-12` 断言，$n=128$ 处失败，相对偏差 $1.8\times10^{-12}$。原因不是推导错，而是 $T_n-1/3$ 属于两个相近数相减：$T_n\approx0.3333$ 的浮点噪声约 $n\varepsilon\sim10^{-14}$ 量级，绝对偏差实测仅 $1.9\times10^{-17}$，但除以 $E_{128}\approx1.0\times10^{-5}$ 后，相对判据被抵消放大了 $|T_n|/|E_n|\approx3\times10^4$ 倍。因此改用 `atol=5e-16`（相对 $T_n$ 约几个 $\varepsilon$，对应求和本身的精度）配 `rtol=1e-10`（留出上述放大因子的余量），两者都远小于 $O(h^2)$ 效应。这一条本身就是第 01 章抵消误差的实例。

对非多项式的光滑函数 $g(x)=e^x$ 只期望 $p=\log_2(E_h/E_{h/2})\to 2$，用 `atol=0.02` 判定，因为有限 $n$ 下还有 $O(h^4)$ 余项。

In [2]:
def trapezoid(f, a, b, n):
    # 复合梯形公式。n 为区间数（正整数）。
    if not isinstance(n, (int, np.integer)) or n < 1:
        raise ValueError("n 必须是正整数")
    x = np.linspace(a, b, n + 1)
    y = f(x)
    h = (b - a) / n
    return h * (y[0] / 2 + y[1:-1].sum() + y[-1] / 2)

def orders(errors):
    # 相邻两次加密的观测收敛阶 log2(E_h / E_{h/2})。
    errors = np.asarray(errors, dtype=float)
    return np.log2(errors[:-1] / errors[1:])

# 开场问题：n 从 4 加倍到 8
e4 = abs(trapezoid(lambda x: x**2, 0.0, 1.0, 4) - 1/3)
e8 = abs(trapezoid(lambda x: x**2, 0.0, 1.0, 8) - 1/3)
print(f"n=4 误差 = {e4:.6e}")
print(f"n=8 误差 = {e8:.6e}")
print(f"误差比 e4/e8 = {e4/e8:.6f}  （预测 4）")

n=4 误差 = 1.041667e-02
n=8 误差 = 2.604167e-03
误差比 e4/e8 = 4.000000  （预测 4）


In [3]:
sizes = np.array([4, 8, 16, 32, 64, 128])

err_sq = np.array([abs(trapezoid(lambda x: x**2, 0.0, 1.0, int(n)) - 1/3) for n in sizes])
exact  = 1.0 / (6 * sizes.astype(float)**2)
np.testing.assert_allclose(err_sq, exact, rtol=1e-10, atol=5e-16)   # 恒等式；容差依据见上方 markdown
print("x^2 与 1/(6n^2) 的最大绝对偏差:", f"{np.max(np.abs(err_sq - exact)):.3e}")
print("最大相对偏差（被抵消放大）:", f"{np.max(np.abs(err_sq - exact) / exact):.3e}")

exp_true = np.e - 1.0
err_exp = np.array([abs(trapezoid(np.exp, 0.0, 1.0, int(n)) - exp_true) for n in sizes])
p_exp = orders(err_exp)
np.testing.assert_allclose(p_exp, 2.0, atol=0.02)       # 渐近阶，允许 O(h^4) 余项

print("n      :", sizes)
print("x^2 误差:", np.array2string(err_sq, formatter={'float_kind': lambda v: f'{v:.3e}'}))
print("1/(6n^2):", np.array2string(exact, formatter={'float_kind': lambda v: f'{v:.3e}'}))
print("e^x 误差:", np.array2string(err_exp, formatter={'float_kind': lambda v: f'{v:.3e}'}))
print("e^x 观测阶:", np.array2string(p_exp, precision=4))

x^2 与 1/(6n^2) 的最大绝对偏差: 1.908e-17
最大相对偏差（被抵消放大）: 1.819e-12
n      : [  4   8  16  32  64 128]
x^2 误差: [1.042e-02 2.604e-03 6.510e-04 1.628e-04 4.069e-05 1.017e-05]
1/(6n^2): [1.042e-02 2.604e-03 6.510e-04 1.628e-04 4.069e-05 1.017e-05]
e^x 误差: [8.940e-03 2.237e-03 5.593e-04 1.398e-04 3.496e-05 8.740e-06]
e^x 观测阶: [1.9989 1.9997 1.9999 2.     2.    ]


## 3. 光滑性失效时会发生什么（一个被实验修正的预判）

梯形误差界 $|E|\le \dfrac{(b-a)h^2}{12}\max|f''|$ 需要 $f\in C^2$。下面用两个都不满足 $C^2$ 的函数，看"阶"会不会掉。

**运行前的预判（A组 g02 记录，后被实验部分推翻）：** 取 $f(x)=|x-1/2|$，$f''$ 在 $x=1/2$ 不存在，我们原本预期观测阶会从 2 降到约 1。

**实验后修正：** 见下方输出——偶数 $n$ 时误差**恰好为 0**，奇数 $n$ 时观测阶仍趋于 2（约 1.70→1.98），只是常数变大。原因是：梯形公式对**分段线性**函数在每个不含折点的小区间上都是精确的；折点在节点上时整体精确，折点落在某个小区间内部时只有**那一个**区间产生 $O(h^2)$ 的误差。所以"不满足 $C^2$"并不自动等于"掉阶"。

**真正会掉阶的例子：** $g(x)=\sqrt{x}$ 在 $[0,1]$，$\int_0^1\sqrt{x}\,dx=2/3$。它在 $x=0$ 处导数无界（端点奇异），理论上复合梯形的误差为 $O(h^{3/2})$。预期观测阶约 1.5。

结论：**阶不是公式的标签**，而是公式、函数光滑性与网格三者配合的结果。

In [4]:
f_kink = lambda x: np.abs(x - 0.5)
true_kink = 0.25   # ∫_0^1 |x-1/2| dx = 1/4

even_n = np.array([4, 8, 16, 32, 64, 128])
odd_n  = np.array([5, 9, 17, 33, 65, 129])

err_even = np.array([abs(trapezoid(f_kink, 0.0, 1.0, int(n)) - true_kink) for n in even_n])
err_odd  = np.array([abs(trapezoid(f_kink, 0.0, 1.0, int(n)) - true_kink) for n in odd_n])

print("偶数 n（折点在节点上）:", np.array2string(err_even, formatter={'float_kind': lambda v: f'{v:.3e}'}))
print("奇数 n（折点在区间内）:", np.array2string(err_odd, formatter={'float_kind': lambda v: f'{v:.3e}'}))
print("奇数序列观测阶（n 近似加倍）:", np.array2string(orders(err_odd), precision=3))
print("同规模对比 n=64 vs n=65: ", f"{err_even[4]:.3e}  vs  {err_odd[4]:.3e}")
print()

# 端点导数奇异：sqrt(x) 在 [0,1]
err_sqrt = np.array([abs(trapezoid(np.sqrt, 0.0, 1.0, int(n)) - 2/3) for n in sizes])
p_sqrt = orders(err_sqrt)
print("sqrt(x) 误差:", np.array2string(err_sqrt, formatter={'float_kind': lambda v: f'{v:.3e}'}))
print("sqrt(x) 观测阶:", np.array2string(p_sqrt, precision=3), " 理论 1.5")
np.testing.assert_allclose(p_sqrt[-1], 1.5, atol=0.05)   # 端点奇异 ⇒ O(h^{3/2})
print("断言通过：|x-1/2| 的阶仍约为 2（常数变大），而 sqrt(x) 掉到约 1.5。")

偶数 n（折点在节点上）: [0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00]
奇数 n（折点在区间内）: [1.000e-02 3.086e-03 8.651e-04 2.296e-04 5.917e-05 1.502e-05]
奇数序列观测阶（n 近似加倍）: [1.696 1.835 1.914 1.956 1.978]
同规模对比 n=64 vs n=65:  0.000e+00  vs  5.917e-05

sqrt(x) 误差: [2.338e-02 8.536e-03 3.085e-03 1.108e-03 3.959e-04 1.410e-04]
sqrt(x) 观测阶: [1.454 1.468 1.478 1.485 1.489]  理论 1.5
断言通过：|x-1/2| 的阶仍约为 2（常数变大），而 sqrt(x) 掉到约 1.5。


In [5]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.6, 4.4))
h = 1.0 / sizes
ax.loglog(h, err_sq, "o-", label=r"$x^2$（误差 $=1/(6n^2)$）")
ax.loglog(h, err_exp, "s-", label=r"$e^x$（光滑）")
ax.loglog(1.0/odd_n, err_odd, "^-", label=r"$|x-1/2|$，$n$ 奇数（阶仍≈2）")
ax.loglog(h, err_sqrt, "d-", label=r"$\sqrt{x}$（端点奇异，阶≈1.5）")
ax.loglog(h, h**2, "k--", lw=1, label=r"参考斜率 $h^2$")
ax.loglog(h, h**1.5, "k:", lw=1, label=r"参考斜率 $h^{1.5}$")
ax.set_xlabel(r"步长 $h=1/n$（无量纲）")
ax.set_ylabel("积分绝对误差（无量纲）")
ax.set_title("复合梯形公式：误差随步长的变化", fontsize=11)
ax.legend(fontsize=7.5)
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
fig.savefig("ABC共用-图片/A-梯形收敛阶.png", dpi=150)
print("已保存 ABC共用-图片/A-梯形收敛阶.png")
plt.close(fig)

已保存 ABC共用-图片/A-梯形收敛阶.png


## 4. 插值 ≠ 在整个区间上逼近

Runge 函数 $f(x)=\dfrac{1}{1+25x^2}$ 在 $[-1,1]$ 上用**等距节点**作多项式插值。插值多项式在节点上误差为 0（这是插值条件），但区间上的最大误差随节点数增加而**增大**（Runge 现象）。

**预期：** 节点上最大误差在 $10^{-14}$ 量级（仅浮点误差）；区间最大误差在 $n$ 较大时明显增长。判据用密网格（4001 点）上的最大误差近似 $\|f-p_n\|_\infty$，这是下界估计而非严格上界。

In [6]:
runge = lambda x: 1.0 / (1.0 + 25.0 * x**2)
dense = np.linspace(-1.0, 1.0, 4001)

def equi_nodes(n):
    return np.linspace(-1.0, 1.0, n + 1)

def cheb_nodes(n):
    # Chebyshev–Lobatto 节点，含端点
    k = np.arange(n + 1)
    return np.cos(k * np.pi / n)

def interp_errors(nodes_fn, n):
    xk = nodes_fn(n)
    coef = np.polyfit(xk, runge(xk), n)       # 次数 n 的插值多项式
    p = np.polyval(coef, dense)
    node_err = np.max(np.abs(np.polyval(coef, xk) - runge(xk)))
    return node_err, np.max(np.abs(p - runge(dense)))

print(f"{'n':>4} | {'等距:节点误差':>14} | {'等距:区间最大误差':>18} | {'Chebyshev:区间最大误差':>22}")
rows = []
for n in [4, 8, 12, 16, 20]:
    ne, me = interp_errors(equi_nodes, n)
    _, mc = interp_errors(cheb_nodes, n)
    rows.append((n, ne, me, mc))
    print(f"{n:>4} | {ne:>14.3e} | {me:>18.3e} | {mc:>22.3e}")

equi_max = np.array([r[2] for r in rows])
cheb_max = np.array([r[3] for r in rows])
assert max(r[1] for r in rows) < 1e-9, "节点上应当几乎零误差"
assert equi_max[-1] > equi_max[1], "等距节点：区间最大误差应随 n 增大"
assert cheb_max[-1] < cheb_max[0], "Chebyshev 节点：区间最大误差应随 n 下降"
print("\n断言通过：节点上误差≈0，但等距节点区间误差随 n 增大，Chebyshev 节点下降。")

   n |        等距:节点误差 |          等距:区间最大误差 |       Chebyshev:区间最大误差
   4 |      2.728e-14 |          4.384e-01 |              4.600e-01
   8 |      5.376e-14 |          1.045e+00 |              2.047e-01
  12 |      2.139e-12 |          3.663e+00 |              8.440e-02
  16 |      5.743e-11 |          1.439e+01 |              3.671e-02
  20 |      4.956e-10 |          5.982e+01 |              1.774e-02

断言通过：节点上误差≈0，但等距节点区间误差随 n 增大，Chebyshev 节点下降。


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0))
n_show = 16
for ax, nodes_fn, name in ((axes[0], equi_nodes, "等距节点"), (axes[1], cheb_nodes, "Chebyshev–Lobatto 节点")):
    xk = nodes_fn(n_show)
    coef = np.polyfit(xk, runge(xk), n_show)
    ax.plot(dense, runge(dense), "k-", lw=1.6, label=r"$f(x)=1/(1+25x^2)$")
    ax.plot(dense, np.polyval(coef, dense), "r-", lw=1.2, label=f"{n_show} 次插值多项式")
    ax.plot(xk, runge(xk), "bo", ms=4, label="插值节点")
    ax.set_title(f"{name}（n={n_show}）")
    ax.set_xlabel("x（无量纲）"); ax.set_ylabel("函数值（无量纲）")
    ax.set_ylim(-0.6, 1.4); ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("ABC共用-图片/A-Runge节点对比.png", dpi=150)
print("已保存 ABC共用-图片/A-Runge节点对比.png")
plt.close(fig)

已保存 ABC共用-图片/A-Runge节点对比.png


## 5. 样条与最小二乘：换一种"逼近"的目标

- **高次插值**：要求过每个节点，代价是区间上可能剧烈振荡。
- **三次样条**：分段三次、二阶连续，误差 $O(h^4)$ 且不会全局振荡。
- **最小二乘多项式**：不要求过节点，最小化残差平方和，用于有噪声或点数远多于自由度的场合。

**预期：** 在等距节点数相同时，三次样条的区间最大误差远小于同节点数的高次插值；低次最小二乘拟合对 Runge 函数仍有明显偏差（它不是插值，也不追求一致逼近）。

In [8]:
from scipy.interpolate import CubicSpline

print(f"{'节点数 n+1':>10} | {'等距高次插值':>14} | {'三次样条':>12} | {'最小二乘(次数6)':>16}")
for n in [8, 16, 20]:
    xk = equi_nodes(n)
    yk = runge(xk)
    poly_err = np.max(np.abs(np.polyval(np.polyfit(xk, yk, n), dense) - runge(dense)))
    spline_err = np.max(np.abs(CubicSpline(xk, yk)(dense) - runge(dense)))
    ls_err = np.max(np.abs(np.polyval(np.polyfit(xk, yk, 6), dense) - runge(dense)))
    print(f"{n+1:>10} | {poly_err:>14.3e} | {spline_err:>12.3e} | {ls_err:>16.3e}")

xk = equi_nodes(20)
assert np.max(np.abs(CubicSpline(xk, runge(xk))(dense) - runge(dense))) < \
       np.max(np.abs(np.polyval(np.polyfit(xk, runge(xk), 20), dense) - runge(dense))), \
       "同节点数下样条应优于 20 次插值"
print("\n断言通过：n=20 时三次样条的区间最大误差小于 20 次等距插值。")

   节点数 n+1 |         等距高次插值 |         三次样条 |        最小二乘(次数6)
         9 |      1.045e+00 |    5.615e-02 |        1.797e-01
        17 |      1.439e+01 |    3.745e-03 |        2.310e-01
        21 |      5.982e+01 |    3.183e-03 |        2.314e-01

断言通过：n=20 时三次样条的区间最大误差小于 20 次等距插值。


## 6. 三种求导：有限差分、符号微分、自动微分

取 $f(x)=e^{\sin x}\sqrt{1+x^2}$，在 $x_0=0.7$ 处求 $f'(x_0)$。

1. **中心差分** $\dfrac{f(x+h)-f(x-h)}{2h}$：截断误差 $O(h^2)$，舍入误差 $O(\varepsilon/h)$，总误差在 $h^\*\sim(3\varepsilon)^{1/3}\approx 6\times10^{-6}$ 附近最小，呈 U 形。
2. **符号微分**（sympy）：给出导数表达式，再代入数值。
3. **前向自动微分**：用对偶数 $a+b\epsilon$（$\epsilon^2=0$）在计算过程中同步传播导数。它**不是**有限差分：没有步长，也没有截断误差。

**预期：** 自动微分与符号导数一致到 $10^{-15}$ 量级；差分最优误差在 $10^{-11}$ 量级，且步长过小时误差反而增大。

In [9]:
import math
from dataclasses import dataclass

@dataclass
class Dual:
    # 前向模式自动微分的对偶数：val + der·ε，ε²=0。
    val: float
    der: float = 0.0

    def __add__(self, o):
        o = o if isinstance(o, Dual) else Dual(o)
        return Dual(self.val + o.val, self.der + o.der)
    __radd__ = __add__

    def __mul__(self, o):
        o = o if isinstance(o, Dual) else Dual(o)
        return Dual(self.val * o.val, self.der * o.val + self.val * o.der)
    __rmul__ = __mul__

    def sin(self):  return Dual(math.sin(self.val), math.cos(self.val) * self.der)
    def exp(self):  return Dual(math.exp(self.val), math.exp(self.val) * self.der)
    def sqrt(self): return Dual(math.sqrt(self.val), self.der / (2 * math.sqrt(self.val)))

def f_generic(x, sin=math.sin, exp=math.exp, sqrt=math.sqrt):
    return exp(sin(x)) * sqrt(1 + x * x)

x0 = 0.7

# 自动微分
xd = Dual(x0, 1.0)
ad = f_generic(xd, sin=lambda t: t.sin(), exp=lambda t: t.exp(), sqrt=lambda t: t.sqrt())

# 符号微分
import sympy as sp
xs = sp.symbols("x")
expr = sp.exp(sp.sin(xs)) * sp.sqrt(1 + xs**2)
d_expr = sp.diff(expr, xs)
sym_val = float(d_expr.subs(xs, sp.Float(x0)))

print("f(x0)        =", f_generic(x0))
print("符号导数表达式:", sp.simplify(d_expr))
print(f"符号导数值   = {sym_val:.16f}")
print(f"自动微分值   = {ad.der:.16f}")
print(f"两者之差     = {abs(ad.der - sym_val):.3e}")
np.testing.assert_allclose(ad.der, sym_val, rtol=1e-13)

f(x0)        = 2.324734286696372


符号导数表达式: (x + (x**2 + 1)*cos(x))*exp(sin(x))/sqrt(x**2 + 1)
符号导数值   = 2.8702119041333467
自动微分值   = 2.8702119041333463
两者之差     = 4.441e-16


In [10]:
hs = np.array([10.0**(-k) for k in range(1, 15)])
central = np.array([(f_generic(x0 + h) - f_generic(x0 - h)) / (2 * h) for h in hs])
fd_err = np.abs(central - sym_val)
best = int(np.argmin(fd_err))

print(f"{'h':>10} | {'中心差分误差':>14}")
for h, e in zip(hs, fd_err):
    print(f"{h:>10.0e} | {e:>14.3e}")
print(f"\n最优步长 h*={hs[best]:.0e}，误差 {fd_err[best]:.3e}")
print(f"理论最优步长 (3eps)^(1/3) = {(3*np.finfo(float).eps)**(1/3):.3e}")
print(f"h=1e-13 时误差 {fd_err[-2]:.3e} 大于最优值 —— 步长越小并非越准")
assert fd_err[best] > abs(ad.der - sym_val), "自动微分应比任何差分步长更准"
print("断言通过：自动微分误差小于全部被测差分步长的误差。")

         h |         中心差分误差
     1e-01 |      5.743e-03
     1e-02 |      5.736e-05
     1e-03 |      5.736e-07
     1e-04 |      5.737e-09
     1e-05 |      5.092e-11
     1e-06 |      1.570e-11
     1e-07 |      4.284e-10
     1e-08 |      8.453e-09
     1e-09 |      5.816e-08
     1e-10 |      1.168e-06
     1e-11 |      1.893e-05
     1e-12 |      1.587e-04
     1e-13 |      1.396e-03
     1e-14 |      1.637e-02

最优步长 h*=1e-06，误差 1.570e-11
理论最优步长 (3eps)^(1/3) = 8.733e-06
h=1e-13 时误差 1.396e-03 大于最优值 —— 步长越小并非越准
断言通过：自动微分误差小于全部被测差分步长的误差。


In [11]:
fig, ax = plt.subplots(figsize=(6.2, 4.2))
ax.loglog(hs, fd_err, "o-", label="中心差分总误差")
ax.loglog(hs, hs**2 * abs(float(sp.diff(expr, xs, 3).subs(xs, sp.Float(x0)))) / 6,
          "k--", lw=1, label=r"截断项 $\propto h^2$")
ax.loglog(hs, np.finfo(float).eps * abs(f_generic(x0)) / hs, "k:", lw=1,
          label=r"舍入项 $\propto \varepsilon/h$")
ax.axhline(max(abs(ad.der - sym_val), 1e-17), color="r", lw=1.4, label="自动微分误差")
ax.set_xlabel("步长 h（无量纲）"); ax.set_ylabel("导数绝对误差（无量纲）")
ax.set_title("中心差分的 U 形误差 vs 自动微分（$x_0=0.7$）", fontsize=11)
ax.legend(fontsize=8); ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
fig.savefig("ABC共用-图片/A-差分步长U形.png", dpi=150)
print("已保存 ABC共用-图片/A-差分步长U形.png")
plt.close(fig)

已保存 ABC共用-图片/A-差分步长U形.png


## 7. 回到开场问题

- **最初的预测**：区间数从 4 加倍到 8，误差变为原来的 $1/4$。
- **支持证据**：第 2 节输出显示误差比恰为 4.000000，且 $E_n$ 与 $1/(6n^2)$ 在 `rtol=1e-12` 下逐项相符——这来自正文第 3 节的恒等式推导，不只是数值巧合。对 $e^x$ 的观测阶收敛到 2。
- **成立条件**：$f\in C^2$ 且网格等距，误差远离舍入极限。第 3 节显示条件失效的方式不止一种：$|x-1/2|$ 的阶仍≈2 但偶数 $n$ 直接精确、奇数 $n$ 常数变大；$\sqrt{x}$ 则真的掉到约 1.5 阶。我们运行前的预判（"$|x-1/2|$ 会掉到 1 阶"）被实验推翻，已在正文与本节改正。
- **比喻的局限**：梯形铺河岸的画面暗示"铺得越细越准"。第 6 节的 U 形曲线说明，在浮点算术里步长过小时舍入误差会反超，越细反而越差；这一点是画面无法表达的。
- **尚未验证**：非等距网格、自适应积分、Gauss 求积的对比；$C^1$ 但非 $C^2$ 函数的阶；多维推广。这些留给后续轮次或 B/C 组提出的问题。

## 8. 参考资料与 AI 使用

参考资料见 [A组-正文.md](A组-正文.md) 末节。本 Notebook 的代码与文字由 A 组（g02）在 Claude Opus 5（Claude Code）辅助下编写，提示词、采纳与拒绝情况记录在 [A组-AI对话记录.md](A组-AI对话记录.md)；人工核验状态见 [A组-编写报告.md](A组-编写报告.md)。上方所有数值都是本 Notebook 在第 1 节所载环境中实际运行的输出。